In [1]:
# setting the environment variables, the keys
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from config import set_environment

# for the keys - as explained early in chapter 2
set_environment()

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Structured output

Let's define the data structure that describes a plan to solve a complex task:

In [3]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate


class Step(BaseModel):
    """A step that is a part of the plan to solve the task."""
    step: str = Field(description="Description of the step")


class Plan(BaseModel):
    """A plan to solve the task."""
    steps: list[Step]


prompt = PromptTemplate.from_template(
    "Prepare a step-by-step plan to solve the give task.\'n"
    "TASK:\n{task}\n"
)

We can explore that the model has succcessfully generated a complex _Pydantic_ structure, and all we needed to do was using a `with_structured_output` method:

In [4]:
result = (prompt | llm.with_structured_output(Plan)).invoke("How to shoot a basketball with good form?")

assert isinstance(result, Plan)
print(f"Amount of steps: {len(result.steps)}")

for idx, step in enumerate(result.steps):
    print(f"STEP {idx+1}: {step.step}\n")

Amount of steps: 7
STEP 1: Find a comfortable, balanced stance with your feet shoulder-width apart, knees slightly bent, and your dominant foot slightly ahead.

STEP 2: Hold the ball with your shooting hand's fingertips spread, forming a 'C' shape, centered under the ball. Your non-shooting hand should be on the side of the ball for balance, not gripping it.

STEP 3: Bring the ball up to your 'shooting pocket' – a position near your chest/shoulder where your elbow is directly under the ball, forming an 'L' shape with your forearm and bicep.

STEP 4: Keep your eyes focused on the front rim of the basket throughout the entire shot.

STEP 5: Extend your shooting arm smoothly upwards, flicking your wrist as the ball leaves your fingertips. Your non-shooting hand should guide the ball until the last moment, then drop away.

STEP 6: Maintain a high 'follow-through' with your shooting hand, as if reaching into a cookie jar on a high shelf, with your index and middle fingers pointing towards t

We can also use a `json_mode` and pass a custom schema to an LLM. 

In [5]:
plan_schema = {
    "type": "ARRAY",
    "items": {
        "type": "OBJECT",
        "properties": {"step": {"type": "STRING"}},
    },
}

query = "How to shoot a basketball with good form?"
result = (prompt | llm.with_structured_output(schema=plan_schema, method="json_mode")).invoke(query)

In [6]:
assert(isinstance(result, list))
print(f"Amount of steps: {len(result)}")
print(result[0])

Amount of steps: 9
{'step': 'Find a comfortable stance with your feet shoulder-width apart, dominant foot slightly forward, and knees slightly bent for balance.'}


In [7]:
result

[{'step': 'Find a comfortable stance with your feet shoulder-width apart, dominant foot slightly forward, and knees slightly bent for balance.'},
 {'step': 'Place your shooting hand under the ball with your fingers spread, and your non-shooting hand lightly on the side of the ball for support.'},
 {'step': "Keep your shooting elbow tucked directly under the ball, forming an 'L' shape with your forearm and bicep."},
 {'step': 'Focus your eyes on the front of the rim throughout the entire shot motion.'},
 {'step': 'Dip slightly with your knees and then extend upwards smoothly, bringing the ball up in a straight line towards the basket.'},
 {'step': 'Push the ball upwards and forwards, extending your shooting arm fully towards the rim.'},
 {'step': 'Release the ball off your fingertips, ensuring a soft touch and spin.'},
 {'step': "Follow through with your shooting hand, letting your wrist snap down to create a 'gooseneck' shape, as if reaching into a cookie jar. Hold this follow-through 

As an alternative, we can use custom arguments supported by the LLM provider. Please, note that these options are vendor-specific, and you need to check the corresponding documentatino for supported extra arguments and formats:

In [30]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI

query = "How to shoot a basketball with good form?"

plan_schema = {
    "name": "plan_schema",
    "description": "Creates a List of steps to achieve a goal",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "steps": {
                "type": "array",
                "description": "List of steps to achieve the goal",
                "items": {
                    "type": "object",
                    "properties": {
                        "step": {
                            "type": "string",
                            "description": "A step to achieve part of the goal",
                        }
                    },
                    "additionalProperties": False,
                    "required": ["step"],
                },
            }
        },
        "additionalProperties": False,
        "required": ["steps"],
    },
}


response_format = {"type": "json_schema", "json_schema": plan_schema}

llm_json = ChatOpenAI(model="gpt-4o-mini", model_kwargs={"response_format":response_format})
result = (prompt | llm_json | JsonOutputParser()).invoke(query)
assert isinstance(Plan.model_validate(result), Plan)
print(f"Amount of steps: {len(result['steps'])}")
print(result['steps'][0])

Amount of steps: 10
{'step': 'Start by positioning yourself at the three-point line or closer to the basket to practice shooting.'}


We can also generated a enum (again, it's a vendor-dependent feature):

In [37]:
from langchain_core.output_parsers import StrOutputParser

response_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "sentiment_classifier",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "sentiment": {
                    "type": "string",
                    "enum": ["positive", "negative", "neutral"],
                }
            },
            "required": ["sentiment"],
            "additionalProperties": False,
        },
    },
}

prompt = PromptTemplate.from_template(
    "Classify the tone of the following customer's review"
    "\n{review}\n"
)

review = "I Like this movie!"
llm_enum = ChatOpenAI(
    model="gpt-4o-mini", model_kwargs={"response_format": response_schema}
)
result = (prompt | llm_enum | JsonOutputParser()).invoke(review)
print(result["sentiment"])

positive
